In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("../shared").resolve()))
sys.path.insert(0, str(Path(".").resolve()))

from constants import (
    ARM_LINK,
    CAPTIONS_PATH,
    CARTESIAN_CONTROL_CACHE_PATH,
    CARTESIAN_CONTROL_RESULTS_PATH,
    CC_A_MAX_EL_NEG,
    CC_A_MAX_EL_POS,
    CC_A_MAX_SH_NEG,
    CC_A_MAX_SH_POS,
    CC_AMP_REG,
    CC_BAND_CHANNELS,
    CC_BAND_SCALE,
    CC_CONSTRAIN_ELBOW,
    CC_DEMO_IDX,
    CC_DEMO_SEED,
    CC_DIST_THRESH,
    CC_EL_HOLD_ALPHA,
    CC_EL_THRESH,
    CC_MIN_PHASE_STEPS,
    CC_MAX_PHASE_STEPS,
    CC_N_R,
    CC_N_SNAPSHOTS,
    CC_N_THETA,
    CC_OPEN_LOOP_OFFSET,
    CC_Q_INT,
    CC_Q_TRACK,
    CC_R_EFFORT,
    CC_SEQ_BLEND_ALPHA,
    CC_SH_DRIFT_THRESH,
    CC_SH_THRESH,
    CC_SH_VEL_DAMP,
    CC_T,
    CC_TARGET_R_MAX,
    CC_TARGET_R_MIN,
    CC_USE_LQI,
    ERA_EM_4_PATH,
    FIGURES_PATH,
    MAX_DELTA,
)
from pgf_utils import (
    apply_figure_style,
    configure_pgf,
    configure_screen,
    figure_inches,
    save_caption,
    save_pgf,
)
from cartesian_control import (
    NOTEBOOK_GITHUB_URL,
    make_cartesian_control_figure,
    print_metrics,
    run_cartesian_control,
)

In [ ]:
# ── 2. Constants ──────────────────────────────────────────────────────────────
# All simulation and figure constants are shared; see notebooks/shared/constants.py.

In [ ]:
# ── 3. Simulation ─────────────────────────────────────────────────────────────
# Loads from cache when constants are unchanged; reruns otherwise.
results, targets, r_grid = run_cartesian_control(
    cache_path=CARTESIAN_CONTROL_CACHE_PATH,
    results_path=CARTESIAN_CONTROL_RESULTS_PATH,
    estimator_path=ERA_EM_4_PATH,
    T=CC_T,
    demo_seed=CC_DEMO_SEED,
    n_r=CC_N_R,
    n_theta=CC_N_THETA,
    target_r_min=CC_TARGET_R_MIN,
    target_r_max=CC_TARGET_R_MAX,
    dist_thresh=CC_DIST_THRESH,
    arm_link=ARM_LINK,
    max_delta=MAX_DELTA,
    q_track=CC_Q_TRACK,
    r_effort=CC_R_EFFORT,
    q_int=CC_Q_INT,
    el_thresh=CC_EL_THRESH,
    sh_thresh=CC_SH_THRESH,
    sh_drift_thresh=CC_SH_DRIFT_THRESH,
    min_phase_steps=CC_MIN_PHASE_STEPS,
    max_phase_steps=CC_MAX_PHASE_STEPS,
    seq_blend_alpha=CC_SEQ_BLEND_ALPHA,
    sh_vel_damp=CC_SH_VEL_DAMP,
    el_hold_alpha=CC_EL_HOLD_ALPHA,
    a_max_sh_pos=CC_A_MAX_SH_POS,
    a_max_sh_neg=CC_A_MAX_SH_NEG,
    a_max_el_pos=CC_A_MAX_EL_POS,
    a_max_el_neg=CC_A_MAX_EL_NEG,
    amp_reg=CC_AMP_REG,
    band_scale=CC_BAND_SCALE,
    constrain_elbow=CC_CONSTRAIN_ELBOW,
    use_lqi=CC_USE_LQI,
    band_channels=CC_BAND_CHANNELS,
    open_loop_offset=CC_OPEN_LOOP_OFFSET,
)
print_metrics(results, T=CC_T, dist_thresh=CC_DIST_THRESH)

In [ ]:
# ── 4. Display ────────────────────────────────────────────────────────────────
configure_screen()
fig = make_cartesian_control_figure(
    results, targets,
    T=CC_T, arm_link=ARM_LINK, dist_thresh=CC_DIST_THRESH,
    n_r=CC_N_R, n_theta=CC_N_THETA, r_grid=r_grid,
    demo_idx=CC_DEMO_IDX, n_snapshots=CC_N_SNAPSHOTS,
    figsize=(16, 10),
)
apply_figure_style(fig)
plt.show()

In [ ]:
# ── 5. PGF export ─────────────────────────────────────────────────────────────
FIG_WIDTH_FRAC  = 1.0
FIG_HEIGHT_FRAC = 0.45

configure_pgf()
fig = make_cartesian_control_figure(
    results, targets,
    T=CC_T, arm_link=ARM_LINK, dist_thresh=CC_DIST_THRESH,
    n_r=CC_N_R, n_theta=CC_N_THETA, r_grid=r_grid,
    demo_idx=CC_DEMO_IDX, n_snapshots=CC_N_SNAPSHOTS,
    figsize=figure_inches(FIG_WIDTH_FRAC, FIG_HEIGHT_FRAC),
)
save_pgf(fig, FIGURES_PATH / "cartesian_control.pgf")
plt.close(fig)

In [ ]:
# ── 6. Caption ────────────────────────────────────────────────────────────────
import numpy as _np

n_reached   = sum(1 for r in results if r["time_to_thresh"] >= 0)
all_ttt     = [r["time_to_thresh"] for r in results if r["time_to_thresh"] >= 0]
all_fd      = [r["dist"][-1] for r in results]
all_frac_in = [r["frac_in_sector"] for r in results]
n_trials    = CC_N_R * CC_N_THETA

caption = (
    r"Cartesian reach evaluation of the sequential LQI controller. "
    f"{n_trials} targets are arranged on a {CC_N_R}\\texttimes{CC_N_THETA} "
    r"polar grid with radii "
    f"$r \\in [{CC_TARGET_R_MIN:.0f}, {CC_TARGET_R_MAX:.0f}]$ cm and angles "
    r"uniformly distributed over $[-\\pi, \\pi)$; each trial runs "
    f"$T={CC_T}$ steps (seed {CC_DEMO_SEED}). "
    r"Top row: distance to target, shoulder error, and elbow error over time; "
    r"individual trials (faint) and mean $\\pm$ 1\\,SD (solid with shading). "
    r"Bottom row (left to right): final distance to target at each grid point; "
    f"example trajectory (trial {CC_DEMO_IDX}) with the optimal sector shaded in "
    r"green and arm snapshots at 6 evenly spaced steps (coloured by time); "
    r"control effort grouped by target radius. "
    f"{n_reached}/{n_trials} trials reached within {CC_DIST_THRESH:.0f}\\,cm "
    f"(mean time $={_np.mean(all_ttt):.0f}$ steps if any reached); "
    f"mean final distance $={_np.mean(all_fd):.1f}$\\,cm; "
    f"mean proportion of steps in optimal sector $={_np.mean(all_frac_in):.2f}$."
)

save_caption(CAPTIONS_PATH / "cartesian_control.tex", caption, NOTEBOOK_GITHUB_URL)
print(caption)

In [ ]:
from cartesian_control import debug_plot_trajectories

debug_plot_trajectories(results, index=CC_DEMO_IDX, targets=targets, dist_thresh=CC_DIST_THRESH)